As análises SQL foram executadas utilizando SQLite, simulando um ambiente real de banco de dados para exploração e validação dos dados.

In [2]:
import sqlite3
import pandas as pd

# carregar dados
df = pd.read_csv("../data/churn.csv")

# criar banco em memória
conn = sqlite3.connect(":memory:")

# salvar dataframe como tabela
df.to_sql("churn", conn, index=False, if_exists="replace")

7043

In [3]:
query = """
SELECT 
    Contract,
    COUNT(*) AS total,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn
FROM churn
GROUP BY Contract
"""

pd.read_sql(query, conn)

,Contract,total,churn
0,Month-to-month,3875,1655
1,One year,1473,166
2,Two year,1695,48


In [ ]:
# Taxa geral de churn
query = """
SELECT
COUNT(*) AS total_clientes,
SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn_clientes,
ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate
FROM churn
"""

pd.read_sql(query, conn)


,total_clientes,churn_clientes,churn_rate
0,7043,122170,26.54


In [5]:
#Churn por tipo de contrato
query = """
SELECT
Contract,
COUNT(*) AS total,
SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churn,
ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate
FROM churn
GROUP BY Contract
ORDER BY churn_rate DESC
"""

pd.read_sql(query, conn)



,Contract,total,churn,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [6]:
#Média de cobrança por churn
query = """
SELECT
Churn,
AVG(MonthlyCharges) AS avg_monthly_charge,
AVG(TotalCharges) AS avg_total_charge
FROM churn
GROUP BY Churn
"""

pd.read_sql(query, conn)




,Churn,avg_monthly_charge,avg_total_charge
0,No,61.265124,2549.911442
1,Yes,74.441332,1531.796094


In [7]:
#Tempo médio de permanência
query = """
SELECT
Churn,
AVG(tenure) AS avg_tenure
FROM churn
GROUP BY Churn
"""

pd.read_sql(query, conn)



,Churn,avg_tenure
0,No,37.569965
1,Yes,17.979133


In [8]:
#Clientes com maior risco (baixo tempo + alto custo)
query = """
SELECT *
FROM churn
WHERE tenure < 12
AND MonthlyCharges > 70
AND Churn = 'Yes'
"""

pd.read_sql(query, conn)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
1,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
2,8168-UQWWF,Female,0,No,No,11,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),97.85,1105.4,Yes
3,7760-OYPDY,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Electronic check,80.65,144.15,Yes
4,7495-OOKFY,Female,1,Yes,No,8,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Credit card (automatic),80.65,633.3,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562,1980-KXVPM,Female,1,No,No,3,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Credit card (automatic),75.05,256.25,Yes
563,0723-DRCLG,Female,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,74.45,74.45,Yes
564,1122-JWTJW,Male,0,Yes,Yes,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,70.65,70.65,Yes
565,6894-LFHLY,Male,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,75.75,75.75,Yes
